In [0]:
# DATADIR = '/datascope/subaru/data/datastore'
DATADIR = '/scratch/aszalay1/dobos/pfs/data/repo/datastore'
RUNDIR = 'u/dobos/dobos-test-123411/20260728T152502Z'
RUN = RUNDIR.replace('/', '_')
DATE = '20250402'
VISIT = '123411'
ARM = 'b'
SPECTROGRAPH = '1'
FIBERID = 130

PFSARM_PATH = f'{DATADIR}/{RUNDIR}/pfsArm/{DATE}/{VISIT}/pfsArm_PFS_{VISIT}_{ARM}{SPECTROGRAPH}_{RUN}.fits'
LINES_PATH = f'{DATADIR}/{RUNDIR}/lines/{DATE}/{VISIT}/lines_PFS_{VISIT}_{ARM}{SPECTROGRAPH}_{RUN}.fits'
SKY_LINES_PATH = '/scratch/aszalay1/dobos/pfs/data/skyLines.txt'

In [0]:
import os
import re
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from astropy.io import fits
from astropy.table import Table

# from pfs.drp.stella import ReferenceLineStatus

class ReferenceLineStatus():
    """Bitmasks for quality of reference lines"""
    GOOD = 0x00 #, "Line is good"
    NOT_VISIBLE = 0x01 #, "Line is not typically visible in PFS (maybe too faint)"
    BLEND = 0x02 #, "Line is blended with other line(s)"
    SUSPECT = 0x04 #, "Line is of suspect quality"
    REJECTED = 0x08 #, "Line has been rejected from use in PFS"
    BROAD = 0x10 #, "Line is broader than normal"
    DETECTORMAP_USED = 0x20 #, "Used for fitting detectorMap"
    DETECTORMAP_RESERVED = 0x40 #, "Reserved during fitting detectorMap"
    SKYSUB_USED = 0x80 #, "Used for 2d sky subtraction"
    MERGED = 0x100 #, "Line has been merged into another"
    COMBINED = 0x200 #, "Line created from MERGED lines"
    PROTECTED = 0x400 #, "Line is protected from discard by exclusion zone"
    LAM_FOCUS = 0x800 #, "Line is used by LAM for detector focus purposes"
    LAM_IMAGEQUALITY = 0x1000 #, "Line is used by LAM for image quality measurement"
    DUPLICATE = 0x2000 #, "Line is a duplicate (e.g. an unmerged component of a merged line)"
    BAD = 0x01F #, "Line is bad for any reason"

# Read the skyLines file

In [0]:
# The file has 5 columns with an arbitrary number of spaces between columns.

sky_lines = pd.read_csv(
    SKY_LINES_PATH, delim_whitespace=True, header=None, comment='#',
    names=['wavelength', 'intensity', 'fwhm', 'line_type', 'comment'])

In [0]:
sky_lines.head(10)

In [0]:
sky_lines.tail(10)

# Load the pfsArm file

In [0]:
with fits.open(PFSARM_PATH) as hdul:
    hdul.info()
    arm_fiberid = hdul['FIBERID'].data
    arm_wave = hdul['WAVELENGTH'].data

arm_fiberid.shape, arm_wave.shape

In [0]:
arm_fiberid

# Load the detected lines file

In [0]:
with fits.open(LINES_PATH) as hdul:
    hdul.info()
    # lines = hdul[''].data
    # lines_description = hdul['ARCLINES_DESCRIPTION'].data
    # lines_transition = hdul['ARCLINES_TRANSITION'].data

In [0]:
lines = Table.read(LINES_PATH, hdu='ARCLINES').to_pandas()
lines

In [0]:
line_description = Table.read(LINES_PATH, hdu='ARCLINES_DESCRIPTION').to_pandas()
line_description['string'] = line_description['string'].map(lambda s: ''.join(s))
line_description

In [0]:
line_description_trace = line_description[line_description['string'] == 'Trace']['number'].item()
line_description_trace

In [0]:
lines_transitions = Table.read(LINES_PATH, hdu='ARCLINES_TRANSITION').to_pandas()
lines_transitions['string'] = lines_transitions['string'].map(lambda s: ''.join(s))
lines_transitions

In [0]:
# Plot all detected lines

fig, ax = plt.subplots(figsize=(12, 12), dpi=240)

# m = (lines.source == 32) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_USED | ReferenceLineStatus.DETECTORMAP_RESERVED)) != 0)

m = (lines.description != line_description_trace) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_USED | ReferenceLineStatus.DETECTORMAP_RESERVED)) != 0)

ax.scatter(
    lines.x[m], lines.y[m], s=1, c=np.log(lines.flux[m] + 1e-3), cmap='inferno',
    alpha=1.0, edgecolor='none', rasterized=True)

In [0]:
# Plot all detected lines

fig, ax = plt.subplots(figsize=(12, 12), dpi=240)

# m = (lines.source == 32) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_RESERVED)) != 0)

m = (lines.description != line_description_trace) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_RESERVED)) != 0)

ax.scatter(
    lines.x[m], lines.y[m], s=1, c=np.log(lines.flux[m] + 1e-3), cmap='inferno',
    alpha=1.0, edgecolor='none', rasterized=True)

In [0]:
# When filtered for the known lines

# m = ((detected_lines.status & (
#         ReferenceLineStatus.DETECTORMAP_USED
#         # | ReferenceLineStatus.DETECTORMAP_RESERVED
#     )) != 0)

m = (lines.description != line_description_trace) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_USED)) != 0)

print(m.sum())
print([hex(x) for x in np.unique(lines.status[m])])
print(np.unique(lines.wavelength[m]).shape)
print(np.unique(lines.wavelength[m]))

# Plot the wavelength solution and the lines

In [0]:
fid = FIBERID

fig, ax = plt.subplots(figsize=(8, 5), dpi=240)

# Wavelength solution from pfsArm
i = np.where(arm_fiberid == fid)[0][0]
y = np.arange(arm_wave.shape[1])
w = arm_wave[i, :]
ip = interp1d(y, w, kind='linear', bounds_error=False, fill_value='extrapolate')

# ax.plot(y, w)
ax.axhline(0, color='k', lw=0.5, alpha=0.5)

# Line data from lines

####

# m = (lines.fiberId == fid) & (lines.source == 32)
# m = (lines.fiberId == fid) & (lines.source == 0)

m = (lines.fiberId == fid) & (lines.description != line_description_trace) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_USED)) != 0)

y = lines.y[m]
y_error = lines.yErr[m]
w = lines.wavelength[m] - ip(y)
w_error = ip(y + y_error) - ip(y - y_error)
c = np.log(lines.flux[m] + 1e-3)

# ax.plot(y, w, '+', markersize=3)
ax.errorbar(y, w, yerr=w_error, fmt='.', markersize=0, elinewidth=0.5)
l = ax.scatter(y, w, c=c, s=8, cmap='jet', alpha=1)

####

m = (lines.fiberId == fid) & (lines.description != line_description_trace) & ((lines.status & (ReferenceLineStatus.DETECTORMAP_RESERVED)) != 0)
print(m.sum())

y = lines.y[m]
y_error = lines.yErr[m]
w = lines.wavelength[m] - ip(y)
w_error = ip(y + y_error) - ip(y - y_error)
c = np.log(lines.flux[m] + 1e-3)

# ax.plot(y, w, '+', markersize=3)
ax.errorbar(y, w, yerr=w_error, fmt='.', markersize=0, elinewidth=0.5)
ax.scatter(y, w, c=c, s=8, cmap='jet', alpha=1, marker='x')

###

ax.set_xlim(0, 4200)
ax.set_ylim(-0.02, 0.02)
# ax.set_ylim(-0.001, 0.001)
ax.set_xlabel('$y$ (pixel)')
ax.set_ylabel(R'$\Delta\lambda$ (nm)')

# Add a secondary x-axis for wavelength based on the wavelength solution from pfsArm
# ip(y) converts pixel to wavelength
ax2 = ax.twiny()
ax2.set_xlim(ip(0), ip(4200))
ax2.set_xlabel(R'$\lambda$ (nm)')

ax.set_title(f'RUN={RUN}\nVISIT={VISIT}, ARM={ARM}, SPECTROGRAPH={SPECTROGRAPH}, FIBERID={FIBERID}')
fig.colorbar(l, ax=ax, label='log flux')

In [0]:
lines.columns

In [0]:
m = (lines.fiberId == fid) & (lines.source == 0)
lines.flag[m]